# Multilingual Handwriting OCR - Google Colab

### Arabic + English + German | TrOCR + EasyOCR + Beam Search

This notebook runs the full [HandwrittenOCR](https://github.com/DrAbdulmalek/HandwrittenOCR) project on Google Colab with GPU acceleration.

**Features:**
- Batch TrOCR inference (3-6x faster)
- Smart Ensemble (TrOCR + EasyOCR)
- Spell checking (EN/AR/DE) with technical keyword protection
- LoRA Fine-tuning + Active Learning
- Gradio UI with 7 tabs
- WER/CER Metrics + Auto Export
- Study Guide generation + Flashcards

---

## Step 1: Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\u2705 GPU Available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("\u26a0\ufe0f No GPU detected. Go to Runtime > Change runtime type > GPU (T4 or better recommended)")

import os
print(f"\nDisk: {os.popen('df -h / | tail -1').read().strip()}")

✅ GPU Available: Tesla T4 (15.6 GB)

Disk: overlay         113G   44G   70G  39% /


## Step 2: Mount Google Drive (Optional)

Mounting Drive allows you to:
- Store processed data permanently
- Cache downloaded models (saves time on future runs)
- Save and load correction dictionaries
- Keep your OCR database synced

In [ ]:
USE_DRIVE = True  # Set to False to run entirely in Colab temp storage

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\u2705 Google Drive mounted successfully")
    PROJECT_ROOT = "/content/drive/MyDrive/Handwritten_OCR_Ultimate"
else:
    PROJECT_ROOT = "/content/Handwritten_OCR_Ultimate"

print(f"Project root: {PROJECT_ROOT}")

Mounted at /content/drive
✅ Google Drive mounted successfully
Project root: /content/drive/MyDrive/Handwritten_OCR_Ultimate


## Step 3: Clone Repository & Install Dependencies

In [ ]:
import os

# Change to a safe directory before potentially removing the current one
try:
    if os.getcwd() != '/content':
        os.chdir('/content')
except FileNotFoundError:
    # If getcwd() fails (likely due to current dir being deleted),
    # force change to a known valid directory.
    print("Warning: Current working directory is invalid. Forcing chdir to /content.")
    os.chdir('/content')

# Clone the project
!rm -rf /content/HandwrittenOCR
!git clone https://github.com/DrAbdulmalek/HandwrittenOCR.git /content/HandwrittenOCR

# Now change into the cloned directory
os.chdir("/content/HandwrittenOCR")

# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-ara > /dev/null 2>&1

# Install Python dependencies
!pip install --quiet -r requirements.txt

print("\u2705 All dependencies installed successfully")

Cloning into '/content/HandwrittenOCR'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 78 (delta 38), reused 78 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 112.93 KiB | 1.85 MiB/s, done.
Resolving deltas: 100% (38/38), done.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 41.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 133.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

## Step 4: Configure Project

In [ ]:
import sys
import os
import json

sys.path.insert(0, "/content/HandwrittenOCR")

# ===== USER SETTINGS =====
HF_TOKEN = ""               # Your Hugging Face token (leave empty for public models)
HF_DATASET_REPO = ""         # e.g., "your-username/handwriting-ocr-data"
MEMORY_MODE = "auto"         # "auto", "low", or "high"
DEFAULT_PDF = "python notes.pdf"  # PDF filename on Drive root, or full path
PAGES_START = 1
PAGES_END = 5
USE_DRIVE = True  # Added USE_DRIVE definition here to prevent NameError
# =========================

from config import Config

# Create config for Colab
if USE_DRIVE:
    config = Config.from_colab_drive(
        pdf_name=DEFAULT_PDF,
        hf_token=HF_TOKEN,
        hf_repo=HF_DATASET_REPO,
        memory_mode=MEMORY_MODE,
    )
else:
    # Run entirely in Colab temp storage (no persistence)
    config = Config(
        project_root="/content/Handwritten_OCR_Ultimate",
        output_dir="/content/Handwritten_OCR_Ultimate",
        pdf_path="/content/input.pdf",
        model_cache_dir="/content/models_cache",
        hf_token=HF_TOKEN,
        hf_dataset_repo=HF_DATASET_REPO,
        memory_mode=MEMORY_MODE,
        gradio_share=True,
        sync_enabled=False,
        easyocr_persistent=False,
    )

config.pages_start = PAGES_START
config.pages_end = PAGES_END

# Full setup: directories, env vars, CSV files, memory mode
config.setup()
config.apply_hf_token()
config.apply_cache_env()
config.setup_easyocr_symlink()

print(f"\u2705 Configuration ready")
print(f"  Project root:   {config.root}")
print(f"  Database:       {config.db_path}")
print(f"  Model cache:    {config.model_cache_dir or config.cache_dir}")
print(f"  Memory mode:    {config.resolve_memory_mode()}")
print(f"  OCR languages:  {config.ocr_languages}")
print(f"  TrOCR model:    {config.trocr_model_name}")
print(f"  Batch size:     {config.trocr_batch_size}")
print(f"  Beam search:    {config.num_beams}")

✅ Configuration ready
  Project root:   /content/drive/MyDrive/Handwritten_OCR_Ultimate
  Database:       /content/drive/MyDrive/Handwritten_OCR_Ultimate/database/handwriting_data.db
  Model cache:    /content/drive/MyDrive/Handwritten_OCR_Ultimate/models_cache
  Memory mode:    high
  OCR languages:  ['en', 'ar', 'de']
  TrOCR model:    David-Magdy/TR_OCR_LARGE
  Batch size:     16
  Beam search:    5


## Step 5: Upload or Select PDF

Choose one of the options below:

In [ ]:
import os
from google.colab import files
from pathlib import Path

# Get the USE_DRIVE status from the configured project root
is_drive_active = config.project_root.startswith("/content/drive")

explicit_file_requested = "python notes.pdf"
expected_drive_path = os.path.join("/content/drive/MyDrive", explicit_file_requested)

# Set the PDF path directly as requested by the user
#config.pdf_path = expected_drive_path
print(f"\u2705 PDF path set to: {config.pdf_path} (as per user request)")

print(f"\n\U0001f4c4 File to process: {config.pdf_path}")

✅ PDF path set to: /content/drive/MyDrive/python notes.pdf (as per user request)

📄 File to process: /content/drive/MyDrive/python notes.pdf


In [ ]:
import os
from google.colab import files
from pathlib import Path

# Get the USE_DRIVE status from the configured project root
is_drive_active = config.project_root.startswith("/content/drive")

print("Choose an option:")
print("  1. Upload a PDF/image from your computer")
print("  2. Use a file from Google Drive")
print("  3. Enter a custom path")

# The 'choice' variable is already set to '2' from the previous execution
# choice = input("\nEnter choice (1/2/3): ").strip()

explicit_file_requested = "python notes.pdf"
expected_drive_path = os.path.join("/content/drive/MyDrive", explicit_file_requested)

# Set the PDF path directly as requested by the user
#config.pdf_path = expected_drive_path

# Store the initial default path for fallback scenarios
initial_config_pdf_path = config.pdf_path

if choice == "1":
    print("\nSelect a PDF or image file...")
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        config.pdf_path = f"/content/{fname}"
        print(f"\u2705 Uploaded: {fname}")
    else:
        print("\u26a0\ufe0f No file uploaded. Using default path from config.")
        config.pdf_path = initial_config_pdf_path # Fallback
elif choice == "2":
    if is_drive_active:
        # Check if the desired file "python notes.pdf" is already configured and exists
        if initial_config_pdf_path == expected_drive_path and os.path.exists(initial_config_pdf_path):
            config.pdf_path = initial_config_pdf_path # Ensure it's set if it's the expected and exists
            print(f"\u2705 The requested file '{os.path.basename(config.pdf_path)}' from Google Drive is already selected.")
            # Skip interactive listing as the user's intent is met by the default config
        else:
            # Fallback to interactive listing if current config.pdf_path is not the explicitly requested one or doesn't exist
            drive_root = "/content/drive/MyDrive"
            pdfs = []
            for f in Path(drive_root).glob("**/*.pdf"):
                pdfs.append(str(f))
            for f in Path(drive_root).glob("**/*.{png,jpg,jpeg,bmp,tiff,webp}"):
                pdfs.append(str(f))
            if pdfs:
                print(f"\nFound {len(pdfs)} files on Drive:")
                for i, p in enumerate(pdfs[:20]):
                    print(f"  [{i+1}] {p.replace(drive_root, '')}")
                sel = input(f"\nEnter number (1-{min(len(pdfs), 20)}): ").strip()
                try:
                    idx = int(sel) - 1
                    config.pdf_path = pdfs[idx]
                    print(f"\u2705 Selected: {config.pdf_path}")
                except (ValueError, IndexError):
                    print("Invalid selection. Using default path from config.")
                    config.pdf_path = initial_config_pdf_path # Fallback
            else:
                print("No PDF/image files found on Drive. Using default path from config.")
                config.pdf_path = initial_config_pdf_path # Fallback
    else:
        print("Google Drive is not set as the project root. Upload a file instead (option 1).")
        print("Using default path from config.")
        config.pdf_path = initial_config_pdf_path # Fallback
elif choice == "3":
    custom_path = input("\nEnter full path to PDF/image: ").strip()
    if os.path.exists(custom_path):
        config.pdf_path = custom_path
        print(f"\u2705 Using: {config.pdf_path}")
    else:
        print(f"\u26a0\ufe0f File not found: {custom_path}")
        print("Using default path from config.")
        config.pdf_path = initial_config_pdf_path # Fallback
else:
    print("Using default path from config.")
    config.pdf_path = initial_config_pdf_path # Fallback

print(f"\n\U0001f4c4 File to process: {config.pdf_path}")

Choose an option:
  1. Upload a PDF/image from your computer
  2. Use a file from Google Drive
  3. Enter a custom path

Found 4 files on Drive:
  [1] /Google AI Studio/IMG_20251228_132616 (1) (1).pdf
  [2] /Google AI Studio/PRO MED.pdf
  [3] /Google AI Studio/IMG_20251228_132616 (1).pdf
  [4] /Google AI Studio/DOC-20251025-WA0000..pdf


In [ ]:
import os
from google.colab import files
from pathlib import Path

# Get the USE_DRIVE status from the configured project root
is_drive_active = config.project_root.startswith("/content/drive")

explicit_file_requested = "python notes.pdf"
expected_drive_path = os.path.join("/content/drive/MyDrive", explicit_file_requested)

# Set the PDF path directly as requested by the user
#config.pdf_path = expected_drive_path
print(f"\u2705 PDF path set to: {config.pdf_path} (as per user request)")

print(f"\n\U0001f4c4 File to process: {config.pdf_path}")

In [ ]:
import os
from google.colab import files
from pathlib import Path

# Get the USE_DRIVE status from the configured project root
is_drive_active = config.project_root.startswith("/content/drive")

explicit_file_requested = "python notes.pdf"
expected_drive_path = os.path.join("/content/drive/MyDrive", explicit_file_requested)

# Set the PDF path directly as requested by the user
#config.pdf_path = expected_drive_path
print(f"\u2705 PDF path set to: {config.pdf_path} (as per user request)")

print(f"\n\U0001f4c4 File to process: {config.pdf_path}")

✅ PDF path set to: /content/drive/MyDrive/python notes.pdf (as per user request)

📄 File to process: /content/drive/MyDrive/python notes.pdf


In [ ]:
import os
from google.colab import files
from pathlib import Path

# Get the USE_DRIVE status from the configured project root
is_drive_active = config.project_root.startswith("/content/drive")

print("Choose an option:")
print("  1. Upload a PDF/image from your computer")
print("  2. Use a file from Google Drive")
print("  3. Enter a custom path")

# The 'choice' variable is already set to '2' from the previous execution
# choice = input("\nEnter choice (1/2/3): ").strip()

explicit_file_requested = "python notes.pdf"
expected_drive_path = os.path.join("/content/drive/MyDrive", explicit_file_requested)

if choice == "1":
    print("\nSelect a PDF or image file...")
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        config.pdf_path = f"/content/{fname}"
        print(f"\u2705 Uploaded: {fname}")
elif choice == "2":
    if is_drive_active:
        # Check if the desired file "python notes.pdf" is already configured
        if config.pdf_path == expected_drive_path and os.path.exists(config.pdf_path):
            print(f"\u2705 The requested file '{os.path.basename(config.pdf_path)}' from Google Drive is already selected.")
            # Skip interactive listing as the user's intent is met by the default config
        else:
            # Fallback to interactive listing if current config.pdf_path is not the explicitly requested one
            drive_root = "/content/drive/MyDrive"
            pdfs = []
            for f in Path(drive_root).glob("**/*.pdf"):
                pdfs.append(str(f))
            for f in Path(drive_root).glob("**/*.{png,jpg,jpeg,bmp,tiff,webp}"):
                pdfs.append(str(f))
            if pdfs:
                print(f"\nFound {len(pdfs)} files on Drive:")
                for i, p in enumerate(pdfs[:20]):
                    print(f"  [{i+1}] {p.replace(drive_root, '')}")
                sel = input(f"\nEnter number (1-{min(len(pdfs), 20)}): ").strip()
                try:
                    idx = int(sel) - 1
                    config.pdf_path = pdfs[idx]
                    print(f"\u2705 Selected: {config.pdf_path}")
                except (ValueError, IndexError):
                    print("Invalid selection. Using default path from config.")
            else:
                print("No PDF/image files found on Drive. Using default path from config.")
    else:
        print("Google Drive is not set as the project root. Upload a file instead (option 1).")
        print("Using default path from config.")
elif choice == "3":
    custom_path = input("\nEnter full path to PDF/image: ").strip()
    if os.path.exists(custom_path):
        config.pdf_path = custom_path
        print(f"\u2705 Using: {config.pdf_path}")
    else:
        print(f"\u26a0\ufe0f File not found: {custom_path}")
        print("Using default path from config.")
else:
    print("Using default path from config.")

print(f"\n\U0001f4c4 File to process: {config.pdf_path}")

Choose an option:
  1. Upload a PDF/image from your computer
  2. Use a file from Google Drive
  3. Enter a custom path

Found 4 files on Drive:
  [1] /Google AI Studio/IMG_20251228_132616 (1) (1).pdf
  [2] /Google AI Studio/PRO MED.pdf
  [3] /Google AI Studio/IMG_20251228_132616 (1).pdf
  [4] /Google AI Studio/DOC-20251025-WA0000..pdf

Enter number (1-4): 3
✅ Selected: /content/drive/MyDrive/Google AI Studio/IMG_20251228_132616 (1).pdf

📄 File to process: /content/drive/MyDrive/Google AI Studio/IMG_20251228_132616 (1).pdf


## Step 6: Load OCR Engine

This downloads/loads the TrOCR and EasyOCR models (first run may take a few minutes).

In [ ]:
import time
import logging

from src.logger import setup_logging
logger = setup_logging(config)

from src.recognition import OCREngine
from src.correction import init_correctors

# Load spell checkers (optional)
if not config.skip_spellcheck:
    print("Loading spell checkers (EN/AR/DE)...")
    init_correctors()
else:
    print("Spell check skipped (low memory mode)")

# Load OCR engine (TrOCR + EasyOCR)
print(f"\nLoading OCR engine...")
print(f"  TrOCR: {config.trocr_model_name}")
print(f"  EasyOCR languages: {config.ocr_languages}")
print(f"  Device: {'CUDA' if config.use_gpu else 'CPU'}")

start = time.time()
ocr_engine = OCREngine(
    trocr_model_name=config.trocr_model_name,
    ocr_languages=config.ocr_languages,
    max_text_length=config.max_text_length,
    cache_dir=config.model_cache_dir or config.cache_dir,
    hf_token=config.hf_token,
    trocr_default_confidence=config.trocr_default_confidence,
    easy_conf_threshold=config.easy_conf_threshold,
    num_beams=config.num_beams,
    trocr_batch_size=config.trocr_batch_size,
    lora_save_path=config.lora_save_path,
    skip_trocr=config.skip_trocr,
)
print(f"\u2705 OCR engine loaded in {time.time() - start:.1f}s")
if ocr_engine.lora_loaded:
    print("\U0001f511 LoRA fine-tuned weights detected and loaded!")

## Step 7: Process PDF / Image

In [ ]:
from src.database import HandwritingDB
from src.pdf_processor import PDFProcessor

# Adjust pages if needed
pages_input = input(f"Process pages {config.pages_start}-{config.pages_end}. Enter custom range (e.g. '1 10') or press Enter: ").strip()
if pages_input:
    parts = pages_input.split()
    if len(parts) == 2:
        config.pages_start = int(parts[0])
        config.pages_end = int(parts[1])

print(f"\nProcessing: {config.pdf_path}")
print(f"Pages: {config.pages_start} to {config.pages_end}")
print(f"DPI: {config.dpi}")
print(f"Batch size: {config.trocr_batch_size}")
print(f"Beam search: {config.num_beams} beams")
print("-" * 50)

# Initialize database
db = HandwritingDB(config.db_path)

# Create processor and run
processor = PDFProcessor(config, ocr_engine, db)

def progress_cb(cur, tot, msg):
    pct = cur / max(tot, 1)
    bar_len = 40
    filled = int(bar_len * pct)
    bar = "\u2588" * filled + "\u2591" * (bar_len - filled)
    print(f"\r  [{bar}] {pct:.0%} - {msg}", end="", flush=True)

stats = processor.process(resume=False, progress_cb=progress_cb)
print()  # newline after progress bar

if stats.get("error"):
    print(f"\n\u26a0\ufe0f Error: {stats['error']}")
else:
    print(f"\n{'='*50}")
    print(f"  Processing Complete")
    print(f"{'='*50}")
    print(f"  Run ID:      {stats.get('run_id', 'N/A')}")
    print(f"  Pages:       {stats.get('pages', 0)}")
    print(f"  Words:       {stats.get('words', 0)}")
    print(f"  Avg conf:    {stats.get('avg_confidence', 0):.2%}")
    print(f"  Duration:    {stats.get('duration_sec', 0):.1f}s")
    print(f"{'='*50}")

## Step 8: View Results

In [ ]:
from src.reconstruction import reconstruct_sentences
from src.database import HandwritingDB

db = HandwritingDB(config.db_path)

# Get all extracted words
words = db.get_all()
print(f"Total words extracted: {len(words) if words else 0}")

# Status breakdown
counts = db.count_by_status()
print(f"\nStatus breakdown:")
for status, count in counts.items():
    print(f"  {status}: {count}")

# Reconstruct sentences
if words:
    print("\n" + "=" * 60)
    print("EXTRACTED TEXT")
    print("=" * 60)
    sentences = reconstruct_sentences(db, verified_only=False)
    if sentences:
        for i, sent in enumerate(sentences[:30], 1):
            lang = sent.get('lang', 'en')
            print(f"\n[P{sent['page']} | {lang}] {sent['text']}")
        if len(sentences) > 30:
            print(f"\n... and {len(sentences) - 30} more sentences")
    else:
        print("\n(Showing raw words - sentence reconstruction needs verified data)")
        for w in words[:30]:
            print(f"  [P{w['page_num']}] {w['predicted_text']} (conf: {w['confidence']:.2f}, src: {w['model_source']})")
        if len(words) > 30:
            print(f"  ... and {len(words) - 30} more words")

## Step 9: Review & Correct (Interactive)

Review OCR results word-by-word and correct mistakes. Your corrections build a learning dictionary that improves future accuracy.

In [ ]:
from src.review_ui import ReviewUI

review_ui = ReviewUI(db, config.feedback_csv)
review_ui.launch()

## Step 10: Sentence Review

In [ ]:
from src.review_ui import SentenceReviewUI

sent_review = SentenceReviewUI(db, config.feedback_csv)
sent_review.launch()

## Step 11: Export Results

In [ ]:
from src.export import auto_export, create_backup
from datetime import datetime

# Auto export: CSV + XLSX + Text + JSONL training data
run_id = f"colab_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
summary = auto_export(db, run_id, config=config)

if summary:
    print(f"\u2705 Export complete:")
    print(f"  Total words: {summary.get('total_words', 0)}")
    print(f"  Verified:     {summary.get('verified', 0)}")
    print(f"  Training:     {summary.get('training_samples', 0)} samples")
    print(f"  Directory:    {summary.get('dir', '')}")
    print(f"\nFiles:")
    for name, path in summary.get('files', {}).items():
        print(f"  [{name}] {path}")

    # Download the CSV
    csv_path = summary.get('files', {}).get('csv', '')
    if csv_path and os.path.exists(csv_path):
        from google.colab import files as colab_files
        print(f"\nDownloading {csv_path}...")
        colab_files.download(csv_path)
else:
    print("\u26a0\ufe0f No data to export")

# Create backup
try:
    backup_path = create_backup(config)
    print(f"\nBackup created: {backup_path}")
except Exception as e:
    print(f"Backup skipped: {e}")

## Step 12: Compute Metrics (WER/CER)

In [ ]:
from src.metrics import compute_metrics, plot_metrics_fig

m = compute_metrics(db, metrics_log=config.metrics_log)

if m.get('wer') is not None:
    print(f"\U0001f4ca Metrics:")
    print(f"  WER:    {m['wer']:.2%}")
    print(f"  CER:    {m['cer']:.2%}")
    print(f"  Samples: {m['samples']}")
else:
    print(f"\u26a0\ufe0f Not enough verified samples with raw_text (need >= 5, have {m.get('samples', 0)})")
    print("Review and correct more words in Step 9 to compute metrics.")

# Plot metrics history
fig = plot_metrics_fig(config.metrics_log)
if fig:
    fig.show()

## Step 13: Fine-Tune with LoRA (Optional)

Fine-tune TrOCR on your verified corrections to improve accuracy. Requires at least 50 verified samples.

In [ ]:
# Check if we have enough verified samples
verified = db.get_verified()
verified_count = len([w for w in verified if w.get('status') in ('verified', 'sentence_corrected')])

print(f"Verified samples available: {verified_count}")

if verified_count >= 50:
    do_finetune = input("Fine-tune now? (y/n): ").strip().lower() == 'y'
    if do_finetune:
        from src.finetuning import finetune_trocr_lora

        success = finetune_trocr_lora(
            ocr_engine=ocr_engine,
            db=db,
            save_path=config.lora_save_path,
            min_samples=config.finetune_min_samples,
            epochs=config.finetune_epochs,
            batch_size=config.finetune_batch_size,
            lr=config.finetune_lr,
            use_fp16=config.use_fp16,
        )
        if success:
            print(f"\u2705 Fine-tuning complete! Model saved to: {config.lora_save_path}")
        else:
            print("\u26a0\ufe0f Fine-tuning failed. Check logs.")
    else:
        print("Fine-tuning skipped.")
else:
    print(f"\u26a0\ufe0f Need at least 50 verified samples (have {verified_count}).")
    print("Review more words in Step 9 first.")

## Step 14: Generate Study Guide (Optional)

Generates a comprehensive study guide with terminology tables, Mermaid diagrams, and flashcards from your handwritten notes.

In [ ]:
from src.study_guide import generate_study_guide_full

print("Generating study guide...")
content = generate_study_guide_full(
    db=db,
    output_dir=config.study_guide_dir,
    include_mermaid=True,
    include_flashcards=True,
)

if content:
    print(f"\u2705 Study guide generated: {config.study_guide_dir}")
    print(f"\nPreview (first 500 chars):")
    print(content[:500])

    # Download study guide
    sg_path = os.path.join(config.study_guide_dir, "study_guide_full.md")
    if os.path.exists(sg_path):
        from google.colab import files as colab_files
        colab_files.download(sg_path)
else:
    print("\u26a0\ufe0f No data for study guide generation.")

## Step 15: Launch Gradio UI

Launch the full-featured Gradio interface with 7 tabs. A public share link will be generated.

In [ ]:
from src.gradio_ui import launch_gradio

# Ensure share is enabled for Colab
config.gradio_share = True

print("\nLaunching Gradio UI...")
print("A public link will appear below when ready.")
print("Open it in a new tab to use the interactive interface.")
print("\n\u26a0\ufe0f To stop: click the Stop button or Runtime > Interrupt execution\n")

launch_gradio(config)

---

## Bonus: Push Training Data to HuggingFace

Upload your verified OCR data as a HuggingFace dataset for sharing or further training.

In [ ]:
if HF_TOKEN and HF_DATASET_REPO:
    from src.export import export_finetuning_dataset, push_to_huggingface

    # Export training dataset
    export_dir = os.path.join(config.exports_dir, "hf_training_dataset")
    result = export_finetuning_dataset(db, export_dir)

    if result:
        print(f"\U0001f4e6 Pushing to HuggingFace: {HF_DATASET_REPO}")
        success = push_to_huggingface(
            local_dataset_dir=export_dir,
            hf_repo_id=HF_DATASET_REPO,
            hf_token=HF_TOKEN,
        )
        if success:
            print(f"\u2705 Dataset pushed to: https://huggingface.co/datasets/{HF_DATASET_REPO}")
    else:
        print("\u26a0\ufe0f No verified data to push.")
else:
    print("\u26a0\ufe0f Set HF_TOKEN and HF_DATASET_REPO in Step 4 to use this feature.")

In [1]:
!apt-get install -y git
!pip install -q zipfile38  # إن احتجت

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
  Preparing metadata (setup.py) ... done


In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
تجميع محتويات جميع مشاريع OCR/NLP العربية مفتوحة المصدر
وتصدير كل مشروع في ملف نصي بالتنسيق التالي:

═══════════════
  مشروع: repo_name.zip
  التصدير: 2026-05-04 12:00:00
  الملفات البرمجية: 45
═══════════════
─────────────────
📄  repo/path/file.py
─────────────────
محتوى الملف ...
"""

import os, sys, json, subprocess, shutil, datetime, zipfile
from pathlib import Path

# قائمة المشاريع من البحث السابق (يمكنك تعديلها/تحديثها)
REPOS = [
    "https://github.com/VikParuchuri/surya",
    "https://github.com/MohamedAliRashad/arabic-nougat",
    "https://github.com/Pythonation/Mistral-Arabic-OCR-test",
    "https://github.com/NAMAA-Space/QARI-OCR",
    "https://github.com/atlasia/AtlasOCR",
    "https://github.com/elly-hacen/Urdu-Arabic-Text-Recognition-AI-Tool",
    "https://github.com/Hedrax/AHDoc",
    "https://github.com/AHR-OCR2024/Arabic-Handwriting-Recognition",
    "https://github.com/PaddlePaddle/PaddleOCR",
    "https://github.com/JaidedAI/EasyOCR",
    "https://github.com/tesseract-ocr/tesseract",
    "https://github.com/microsoft/unilm",            # TrOCR ضمنه
    "https://github.com/microsoft/table-transformer",
    "https://github.com/uobinxiao/SparseTableDet",
    "https://github.com/ShakilMahmudShuvo/Borderless-Tables-Detection",
    "https://github.com/holmeswww/PdfTable",
    "https://github.com/tabulapdf/tabula-java",
    "https://github.com/camelot-dev/camelot",
    "https://github.com/DS4SD/docling",
    "https://github.com/qurator-spk/eynollah",
    "https://github.com/Layout-Parser/layout-parser",
    "https://github.com/HURIDOCS/pdf-document-layout-analysis",
    "https://github.com/OCR4all/LAREX",
    "https://github.com/AhmedFarouk04/Faseeh-Arabic-NLP",
    "https://github.com/khaled-rashwani/Arabic-Spelling-Correction",
    "https://github.com/Jawdat-Kadour/Arabic-Spelling-Correction-Using-Context-Aware-Neural-Networks",
    "https://github.com/basselkassem/ar_corrector",
    "https://github.com/adhaamehab/textblob-ar",
    "https://github.com/SINAI-Lab/SinaTools",
    "https://github.com/stanfordnlp/stanza",
    "https://github.com/ARBML/ARBML",
    "https://github.com/dolphin2003/arabic-nlp-toolkit",
    "https://github.com/egdels/makeacopy",
    "https://github.com/aiviewz/Synthdog-RTL",
    "https://github.com/Huscs/Arabic-PDF-OCR",
    "https://github.com/andbue/nashi",
    "https://github.com/datalab-to/chandra-ocr",
    "https://github.com/Ethan-q/pdf-master",
    "https://github.com/CatchTheTornado/text-extract-api",
    "https://github.com/GNOME/ocrfeeder",
    "https://github.com/citizenbob/ml-model-training-ui",
    "https://github.com/revdoku/revdoku",
    "https://github.com/WVHIBV/Ocr_Project",
    "https://github.com/MBZUAI-oryx/KITAB-Bench",
    "https://github.com/calfa-co/rasam-dataset",
]

# صيغ الملفات التي سنضمّنها (كلها نصية، يمكن قراءتها)
TEXT_EXTENSIONS = {
    ".py", ".sh", ".txt", ".md", ".rst", ".cfg", ".ini", ".toml",
    ".yaml", ".yml", ".json", ".xml", ".html", ".css", ".js", ".ts",
    ".jsx", ".tsx", ".c", ".cpp", ".h", ".hpp", ".java", ".kt",
    ".swift", ".rs", ".go", ".rb", ".php", ".pl", ".lua", ".r",
    ".jl", ".ex", ".exs", ".dart", ".bash", ".zsh", ".fish",
    ".cfg", ".conf", ".env", ".gitignore", ".dockerfile", ".makefile",
    ".ipynb", ".tex", ".cls", ".sty",
}

# المجلد المؤقت للعمل
BASE = Path.home() / "ocr_projects_export_temp"
OUTPUT_DIR = Path.home() / "projects_formatted"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def repo_name(url):
    """استخراج اسم المستودع من الرابط"""
    return url.rstrip("/").split("/")[-1].replace(".git", "")

def clone_repo(url, dest):
    """استنساخ خفيف (أحدث commit فقط) لتوفير الوقت والمساحة"""
    if dest.exists():
        print(f"⏩ المجلد {dest} موجود مسبقاً، تخطي الاستنساخ...")
        return True
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", url, str(dest)],
            check=True, capture_output=True, text=True
        )
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ فشل استنساخ {url}: {e.stderr}")
        return False

def count_and_print_files(root):
    """
    إحصاء الملفات النصية داخل مجلد root،
    وإرجاع قائمة مساراتها النسبية.
    """
    files = []
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in TEXT_EXTENSIONS:
            files.append(path.relative_to(root))
    return files

def generate_project_file(repo_url, repo_dir, output_dir):
    """
    اكتُب محتويات المشروع في ملف نصي واحد بالتنسيق المطلوب.
    """
    name = repo_name(repo_url)
    rel_files = count_and_print_files(repo_dir)
    total = len(rel_files)
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    out_path = output_dir / f"{name}.txt"
    with open(out_path, "w", encoding="utf-8") as out:
        # الرأس
        out.write("═" * 70 + "\n")
        out.write(f"  مشروع: {name}.zip\n")
        out.write(f"  التصدير: {now}\n")
        out.write(f"  الملفات البرمجية: {total}\n")
        out.write("═" * 70 + "\n")

        # محتوى كل ملف
        for rel in rel_files:
            full = repo_dir / rel
            try:
                content = full.read_text(encoding="utf-8", errors="replace")
            except Exception:
                content = "[ملف ثنائي أو غير قابل للقراءة]"
            out.write("\n" + "─" * 70 + "\n")
            out.write(f"📄  {name}/{rel}\n")
            out.write("─" * 70 + "\n")
            out.write(content)
            out.write("\n")
    return out_path

def main():
    print("🔄 بدء جمع المشاريع...")
    BASE.mkdir(exist_ok=True)
    generated = []

    for url in REPOS:
        name = repo_name(url)
        repo_dir = BASE / name
        print(f"\n📥 معالجة {name} ...")
        if not clone_repo(url, repo_dir):
            continue
        try:
            out_file = generate_project_file(url, repo_dir, OUTPUT_DIR)
            generated.append(out_file)
            print(f"✅ تم تصدير: {out_file}")
        except Exception as e:
            print(f"⚠️ خطأ في معالجة {name}: {e}")

    # ضغط النتائج
    zip_path = OUTPUT_DIR.with_suffix(".zip")
    with zipfile.ZipFile(zip_path, "w") as zf:
        for f in generated:
            zf.write(f, arcname=f.name)
    print(f"\n🎉 تم تجميع {len(generated)} مشروعاُ في الملف المضغوط: {zip_path}")

if __name__ == "__main__":
    main()

🔄 بدء جمع المشاريع...

📥 معالجة surya ...
✅ تم تصدير: /root/projects_formatted/surya.txt

📥 معالجة arabic-nougat ...
✅ تم تصدير: /root/projects_formatted/arabic-nougat.txt

📥 معالجة Mistral-Arabic-OCR-test ...
✅ تم تصدير: /root/projects_formatted/Mistral-Arabic-OCR-test.txt

📥 معالجة QARI-OCR ...
❌ فشل استنساخ https://github.com/NAMAA-Space/QARI-OCR: Cloning into '/root/ocr_projects_export_temp/QARI-OCR'...
fatal: could not read Username for 'https://github.com': No such device or address


📥 معالجة AtlasOCR ...
❌ فشل استنساخ https://github.com/atlasia/AtlasOCR: Cloning into '/root/ocr_projects_export_temp/AtlasOCR'...
fatal: could not read Username for 'https://github.com': No such device or address


📥 معالجة Urdu-Arabic-Text-Recognition-AI-Tool ...
✅ تم تصدير: /root/projects_formatted/Urdu-Arabic-Text-Recognition-AI-Tool.txt

📥 معالجة AHDoc ...
✅ تم تصدير: /root/projects_formatted/AHDoc.txt

📥 معالجة Arabic-Handwriting-Recognition ...
✅ تم تصدير: /root/projects_formatted/Arabic-Hand

In [3]:
from google.colab import files
files.download('/root/projects_formatted.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
failed_repos = [
    "https://github.com/NAMAA-Space/QARI-OCR",
    "https://github.com/atlasia/AtlasOCR",
    "https://github.com/holmeswww/PdfTable",
    "https://github.com/SINAI-Lab/SinaTools",
    "https://github.com/datalab-to/chandra-ocr"
]
import subprocess, os
os.chdir('/root/ocr_projects_export_temp')
for url in failed_repos:
    name = url.rstrip('/').split('/')[-1].replace('.git','')
    print(f'محاولة {name}...')
    subprocess.run(['git', 'clone', '--depth', '1', url],
                  stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)

محاولة QARI-OCR...
محاولة AtlasOCR...
محاولة PdfTable...
محاولة SinaTools...
محاولة chandra-ocr...


In [5]:
import subprocess, datetime, zipfile
from pathlib import Path

# المجلدات نفسها من السكربت السابق
BASE = Path.home() / "ocr_projects_export_temp"
OUTPUT_DIR = Path.home() / "projects_formatted"

# المشاريع التي أعدنا استنساخها
extra_repos = [
    "https://github.com/NAMAA-Space/QARI-OCR",
    "https://github.com/atlasia/AtlasOCR",
    "https://github.com/holmeswww/PdfTable",
    "https://github.com/SINAI-Lab/SinaTools",
    "https://github.com/datalab-to/chandra-ocr",
]

# نعيد تعريف الدوال المطلوبة فقط (نسختها السابقة تعمل، لكن يمكن استدعاؤها)
def repo_name(url):
    return url.rstrip("/").split("/")[-1].replace(".git", "")

def count_and_print_files(root):
    TEXT_EXTENSIONS = {
        ".py", ".sh", ".txt", ".md", ".rst", ".cfg", ".ini", ".toml",
        ".yaml", ".yml", ".json", ".xml", ".html", ".css", ".js", ".ts",
        ".jsx", ".tsx", ".c", ".cpp", ".h", ".hpp", ".java", ".kt",
        ".swift", ".rs", ".go", ".rb", ".php", ".pl", ".lua", ".r",
        ".jl", ".ex", ".exs", ".dart", ".bash", ".zsh", ".fish",
        ".cfg", ".conf", ".env", ".gitignore", ".dockerfile", ".makefile",
        ".ipynb", ".tex", ".cls", ".sty",
    }
    files = []
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in TEXT_EXTENSIONS:
            files.append(path.relative_to(root))
    return files

def generate_project_file(repo_url, repo_dir, output_dir):
    name = repo_name(repo_url)
    rel_files = count_and_print_files(repo_dir)
    total = len(rel_files)
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    out_path = output_dir / f"{name}.txt"
    with open(out_path, "w", encoding="utf-8") as out:
        out.write("═" * 70 + "\n")
        out.write(f"  مشروع: {name}.zip\n")
        out.write(f"  التصدير: {now}\n")
        out.write(f"  الملفات البرمجية: {total}\n")
        out.write("═" * 70 + "\n")
        for rel in rel_files:
            full = repo_dir / rel
            try:
                content = full.read_text(encoding="utf-8", errors="replace")
            except:
                content = "[ملف غير قابل للقراءة]"
            out.write("\n" + "─" * 70 + "\n")
            out.write(f"📄  {name}/{rel}\n")
            out.write("─" * 70 + "\n")
            out.write(content)
            out.write("\n")
    return out_path

# توليد ملفات المشاريع المتبقية
new_files = []
for url in extra_repos:
    name = repo_name(url)
    repo_dir = BASE / name
    if repo_dir.exists():
        out_file = generate_project_file(url, repo_dir, OUTPUT_DIR)
        new_files.append(out_file)
        print(f"✅ تم تصدير: {out_file.name}")
    else:
        print(f"⚠️ المجلد {name} غير موجود رغم المحاولة")

# تحديث الملف المضغوط بإضافة الملفات الجديدة
zip_path = OUTPUT_DIR.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "a") as zf:  # وضع 'a' للإضافة على الموجود
    for f in new_files:
        zf.write(f, arcname=f.name)

print(f"🎉 تمت إضافة {len(new_files)} مشروع إلى الملف المضغوط: {zip_path}")

⚠️ المجلد QARI-OCR غير موجود رغم المحاولة
⚠️ المجلد AtlasOCR غير موجود رغم المحاولة
⚠️ المجلد PdfTable غير موجود رغم المحاولة
⚠️ المجلد SinaTools غير موجود رغم المحاولة
⚠️ المجلد chandra-ocr غير موجود رغم المحاولة
🎉 تمت إضافة 0 مشروع إلى الملف المضغوط: /root/projects_formatted.zip


In [6]:
import subprocess
from pathlib import Path

BASE = Path.home() / "ocr_projects_export_temp"

failed_repos = {
    "QARI-OCR": "https://github.com/NAMAA-Space/QARI-OCR",
    "AtlasOCR": "https://github.com/atlasia/AtlasOCR",
    "PdfTable": "https://github.com/holmeswww/PdfTable",
    "SinaTools": "https://github.com/SINAI-Lab/SinaTools",
    "chandra-ocr": "https://github.com/datalab-to/chandra-ocr",
}

for name, url in failed_repos.items():
    dest = BASE / name
    if dest.exists():
        print(f"✅ {name} موجود بالفعل")
        continue
    print(f"🔄 محاولة استنساخ {name}...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", url, str(dest)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✅ نجح")
    else:
        print(f"❌ فشل: {result.stderr.strip()}")

🔄 محاولة استنساخ QARI-OCR...
❌ فشل: Cloning into '/root/ocr_projects_export_temp/QARI-OCR'...
fatal: could not read Username for 'https://github.com': No such device or address
🔄 محاولة استنساخ AtlasOCR...
❌ فشل: Cloning into '/root/ocr_projects_export_temp/AtlasOCR'...
fatal: could not read Username for 'https://github.com': No such device or address
🔄 محاولة استنساخ PdfTable...
❌ فشل: Cloning into '/root/ocr_projects_export_temp/PdfTable'...
fatal: could not read Username for 'https://github.com': No such device or address
🔄 محاولة استنساخ SinaTools...
❌ فشل: Cloning into '/root/ocr_projects_export_temp/SinaTools'...
fatal: could not read Username for 'https://github.com': No such device or address
🔄 محاولة استنساخ chandra-ocr...
❌ فشل: Cloning into '/root/ocr_projects_export_temp/chandra-ocr'...
fatal: could not read Username for 'https://github.com': No such device or address


In [7]:
import requests, zipfile, io

def download_zip(url, dest_folder):
    # url: https://github.com/user/repo
    zip_url = url.rstrip('/') + '/archive/refs/heads/main.zip'
    response = requests.get(zip_url)
    if response.status_code == 200:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extractall(dest_folder.parent)
        # المجلد المستخرج يكون بصيغة repo-main، نعيد تسميته
        repo_name = url.rstrip('/').split('/')[-1]
        extracted = dest_folder.parent / f"{repo_name}-main"
        if extracted.exists():
            extracted.rename(dest_folder)
            return True
    return False

# تطبيقها على المشاريع المتبقية
for name, url in failed_repos.items():
    dest = BASE / name
    if not dest.exists():
        print(f"📥 تحميل {name} كأرشيف ZIP...")
        if download_zip(url, dest):
            print(f"✅ تم تحميل {name}")
        else:
            print(f"❌ تعذّر تحميل {name} (قد يكون المستودع خاصًا أو محذوفًا)")

📥 تحميل QARI-OCR كأرشيف ZIP...
❌ تعذّر تحميل QARI-OCR (قد يكون المستودع خاصًا أو محذوفًا)
📥 تحميل AtlasOCR كأرشيف ZIP...
❌ تعذّر تحميل AtlasOCR (قد يكون المستودع خاصًا أو محذوفًا)
📥 تحميل PdfTable كأرشيف ZIP...
❌ تعذّر تحميل PdfTable (قد يكون المستودع خاصًا أو محذوفًا)
📥 تحميل SinaTools كأرشيف ZIP...
❌ تعذّر تحميل SinaTools (قد يكون المستودع خاصًا أو محذوفًا)
📥 تحميل chandra-ocr كأرشيف ZIP...
❌ تعذّر تحميل chandra-ocr (قد يكون المستودع خاصًا أو محذوفًا)
